# Semantic Search FAQ Assistant
**Semantic Search & LLM Application: E-Commerce Customer Service**

Notebook ini membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

| Tahap | Implementasi di notebook ini |
|---|---|
| **Ingest** | Section 1–2: load CSV dengan Pandas, eksplorasi struktur, gabungkan question + answer menjadi unit retrievable |
| **Embed** | Section 3: `gemini-embedding-001` (task type `RETRIEVAL_DOCUMENT` untuk dokumen, `RETRIEVAL_QUERY` untuk query) |
| **Store** | Section 4: FAISS `IndexFlatIP` atas vektor ter-normalisasi (inner product = cosine similarity), disimpan ke `vector_store/` |
| **Retrieve** | Section 5: `semantic_search(query, top_k)` → top-k FAQ + similarity score |
| **Generate** | Section 6–7: prompt terstruktur (Persona, Task, Context, Format) → Gemini LLM |

## 1. Import Library & Load Knowledge Base

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
import numpy as np
import faiss
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors as genai_errors

# API key dibaca dari file .env (GEMINI_API_KEY=...), tidak di-hardcode di notebook
load_dotenv()
client = genai.Client(
    api_key=os.environ.get('GEMINI_API_KEY'),
    http_options=types.HttpOptions(timeout=60_000),  # timeout 60 detik per request agar tidak hang
)

# Path project: notebook dijalankan dari folder notebook/ maupun dari root project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'ecommerce_faq_knowledge_base.csv'
VECTOR_DIR = PROJECT_ROOT / 'vector_store'
VECTOR_DIR.mkdir(exist_ok=True)

# Konfigurasi model
EMBEDDING_MODEL = 'gemini-embedding-001'
# Urutan model LLM yang dicoba; jika satu model kena limit kuota / tidak tersedia, pindah ke model berikutnya
LLM_MODELS = ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite']

df = pd.read_csv(DATA_PATH)
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


### Eksplorasi struktur knowledge base
Memeriksa kolom, tipe data, missing value, duplikat, distribusi kategori, dan panjang teks sebelum dijadikan unit retrievable. Data asli **tidak diubah**; semua transformasi dilakukan pada DataFrame di memori.

In [2]:
print('Shape:', df.shape)
print('Kolom:', df.columns.tolist())
print()
df.info()
print()
print('Missing value per kolom:', df.isna().sum().to_dict())
print('Duplikat question:', df['question'].duplicated().sum())
print('faq_id unik:', df['faq_id'].is_unique)

Shape: (59, 4)
Kolom: ['faq_id', 'category', 'question', 'answer']

<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   faq_id    59 non-null     int64
 1   category  59 non-null     str  
 2   question  59 non-null     str  
 3   answer    59 non-null     str  
dtypes: int64(1), str(3)
memory usage: 2.0 KB

Missing value per kolom: {'faq_id': 0, 'category': 0, 'question': 0, 'answer': 0}
Duplikat question: 0
faq_id unik: True


In [3]:
print('Jumlah kategori:', df['category'].nunique())
df['category'].value_counts().rename_axis('category').to_frame('jumlah_entri')

Jumlah kategori: 8


,jumlah_entri
category,
Pemesanan,9
Pengiriman,9
Pembayaran,8
Pengembalian,8
Akun,7
Produk,7
Promo,6
Customer Service,5


In [4]:
text_len = pd.DataFrame({
    'question_words': df['question'].str.split().str.len(),
    'answer_words': df['answer'].str.split().str.len(),
})
text_len.describe().round(1)

,question_words,answer_words
count,59.0,59.0
mean,7.6,17.2
std,1.5,2.6
min,5.0,11.0
25%,6.5,15.0
50%,8.0,17.0
75%,9.0,19.0
max,11.0,23.0


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

Kategori juga ikut disertakan karena memberi sinyal topik tambahan (misal "Pengembalian") yang membantu query pendek/ambigu. Karena setiap entri pendek (± 10–35 kata), satu entri = satu unit; chunking tidak diperlukan.

In [5]:
df['content'] = (
    'Kategori: ' + df['category'] +
    '\nPertanyaan: ' + df['question'] +
    '\nJawaban: ' + df['answer']
)
documents = df['content'].tolist()
print('Jumlah unit retrievable:', len(documents))
print()
print('Contoh unit retrievable:')
print(documents[0])

Jumlah unit retrievable: 59

Contoh unit retrievable:
Kategori: Pemesanan
Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

- Model: `gemini-embedding-001` (model `text-embedding-004` pada starter sudah tidak tersedia di API).
- `task_type='RETRIEVAL_DOCUMENT'` untuk entri FAQ dan `task_type='RETRIEVAL_QUERY'` untuk pertanyaan pengguna. Gemini mengoptimasi embedding secara asimetris untuk kasus retrieval (query pendek vs dokumen yang lebih panjang).
- Embedding dikirim per batch (bukan satu per satu) sehingga 59 entri cukup 1–2 request API.
- Vektor dinormalisasi (L2 norm = 1) agar inner product di FAISS setara dengan **cosine similarity**.

In [6]:
def with_retry(fn, max_retries=5, base_wait=10):
    """Memanggil fn() dan mengulang jika terkena rate limit (429) atau server error (5xx)."""
    for attempt in range(max_retries):
        try:
            return fn()
        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            retryable = getattr(e, 'code', None) == 429 or isinstance(e, genai_errors.ServerError)
            if not retryable or attempt == max_retries - 1:
                raise
            wait = base_wait * (attempt + 1)
            print(f'  [retry] {getattr(e, "code", "")} - menunggu {wait} detik...')
            time.sleep(wait)


def get_embeddings(texts, task_type, batch_size=50):
    """Mengubah list teks menjadi matriks embedding (n_texts, dim) float32 yang sudah dinormalisasi."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        result = with_retry(lambda: client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(task_type=task_type),
        ))
        vectors.extend(e.values for e in result.embeddings)
    vectors = np.array(vectors, dtype='float32')
    faiss.normalize_L2(vectors)  # in-place, agar inner product = cosine similarity
    return vectors


def get_embedding(text, task_type='RETRIEVAL_QUERY'):
    """Embedding untuk satu teks (dipakai untuk query pengguna). Return shape (1, dim)."""
    return get_embeddings([text], task_type=task_type)


embeddings = get_embeddings(documents, task_type='RETRIEVAL_DOCUMENT')
print('Shape embeddings:', embeddings.shape)
print('Dimensi per vektor:', embeddings.shape[1])
print('Norm vektor (harus ~1.0):', np.round(np.linalg.norm(embeddings, axis=1)[:5], 4))

assert embeddings.shape[0] == len(df), 'Jumlah embedding tidak sama dengan jumlah entri FAQ!'

Shape embeddings: (59, 3072)
Dimensi per vektor: 3072
Norm vektor (harus ~1.0): [1. 1. 1. 1. 1.]


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

Menggunakan `IndexFlatIP` (exact search, inner product). Karena vektor sudah dinormalisasi, skor yang dihasilkan adalah cosine similarity: makin mendekati 1 makin mirip. Untuk 59 entri, exact search (brute force) sudah sangat cepat; index aproksimasi (IVF/HNSW) baru relevan untuk jutaan vektor.

`IndexIDMap` dipakai agar ID di index = `faq_id` asli, sehingga hasil pencarian bisa langsung dipetakan ke entri knowledge base.

In [7]:
dimension = embeddings.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(dimension))
index.add_with_ids(embeddings, df['faq_id'].to_numpy().astype('int64'))

print('Dimensi index:', index.d)
print('Jumlah vektor tersimpan di index:', index.ntotal)
print('Jumlah entri knowledge base     :', len(df))
assert index.ntotal == len(df), 'Jumlah vektor di FAISS tidak sesuai jumlah entri!'
print('Verifikasi OK: semua entri FAQ tersimpan di vector database.')

Dimensi index: 3072
Jumlah vektor tersimpan di index: 59
Jumlah entri knowledge base     : 59
Verifikasi OK: semua entri FAQ tersimpan di vector database.


Index FAISS disimpan ke disk (`vector_store/faq_index.faiss`) beserta metadata FAQ-nya, sehingga dapat dimuat ulang tanpa memanggil Embeddings API lagi.

In [8]:
INDEX_PATH = VECTOR_DIR / 'faq_index.faiss'
META_PATH = VECTOR_DIR / 'faq_metadata.csv'

faiss.write_index(index, str(INDEX_PATH))
df[['faq_id', 'category', 'question', 'answer']].to_csv(META_PATH, index=False)

# Verifikasi: muat ulang index dari disk
loaded_index = faiss.read_index(str(INDEX_PATH))
print('Index tersimpan di :', INDEX_PATH.relative_to(PROJECT_ROOT))
print('Vektor di index yang dimuat ulang:', loaded_index.ntotal)

Index tersimpan di : vector_store\faq_index.faiss
Vektor di index yang dimuat ulang: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

Alur: query → embedding (`RETRIEVAL_QUERY`) → cari top-k di FAISS → kembalikan entri FAQ + similarity score.

In [9]:
faq_lookup = df.set_index('faq_id')


def semantic_search(query, top_k=3):
    query_embedding = get_embedding(query, task_type='RETRIEVAL_QUERY')
    scores, ids = index.search(query_embedding, top_k)

    results = []
    for rank, (faq_id, score) in enumerate(zip(ids[0], scores[0]), start=1):
        if faq_id == -1:  # FAISS mengembalikan -1 jika hasil < top_k
            continue
        row = faq_lookup.loc[faq_id]
        results.append({
            'rank': rank,
            'faq_id': int(faq_id),
            'category': row['category'],
            'question': row['question'],
            'answer': row['answer'],
            'similarity': round(float(score), 4),
        })
    return results


def show_results(query, results):
    print(f'Query: {query}')
    return pd.DataFrame(results)[['rank', 'faq_id', 'category', 'question', 'similarity']]


# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
show_results('kenapa paket saya belum sampai?', test_results)

Query: kenapa paket saya belum sampai?


,rank,faq_id,category,question,similarity
0,1,12,Pengiriman,Bagaimana jika paket saya belum sampai melebih...,0.7971
1,2,14,Pengiriman,Kenapa status pengiriman saya tidak berubah se...,0.7816
2,3,52,Pengiriman,Kenapa nomor resi saya belum bisa dilacak di w...,0.7593


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

Prinsip desain:
- **Persona**: asisten CS e-commerce yang ramah, berbahasa Indonesia.
- **Task**: menjawab **hanya** dari context; dilarang memakai pengetahuan umum atau mengarang angka/kebijakan.
- **Context**: top-k FAQ hasil retrieval, diberi nomor sumber dan similarity score. LLM diberi tahu bahwa tidak semua FAQ pasti relevan, jadi ia harus memilih sendiri.
- **Format**: jawaban singkat 2–4 kalimat / poin langkah, plus penanda sumber (`[FAQ #id]`) agar jawaban bisa diaudit. Jika informasi tidak ada → kalimat penolakan baku + arahkan ke customer service.

In [10]:
PROMPT_TEMPLATE = '''### PERSONA
Kamu adalah "Asisten CS", asisten virtual customer service sebuah platform e-commerce di Indonesia.
Gaya bahasamu ramah, sopan, jelas, dan menggunakan Bahasa Indonesia yang mudah dipahami pelanggan.

### TASK
Jawab pertanyaan pelanggan HANYA berdasarkan informasi pada bagian CONTEXT di bawah.
Aturan:
1. Gunakan hanya fakta yang tertulis di CONTEXT. Jangan menambahkan informasi dari pengetahuan umum,
   jangan mengarang angka, durasi, biaya, nama fitur, atau kebijakan yang tidak ada di CONTEXT.
2. CONTEXT berisi beberapa FAQ hasil pencarian otomatis dan tidak semuanya pasti relevan.
   Pilih hanya FAQ yang benar-benar menjawab pertanyaan; abaikan yang tidak relevan.
3. Jika CONTEXT tidak memuat informasi untuk menjawab pertanyaan (atau hanya membahas topik yang mirip
   tetapi tidak menjawab pertanyaannya), JANGAN menebak. Jawab dengan:
   "Mohon maaf, saya belum memiliki informasi mengenai hal tersebut di FAQ kami."
   lalu sarankan pelanggan menghubungi customer service.
4. Jika pertanyaan hanya sebagian terjawab, jawab bagian yang ada di CONTEXT dan sebutkan bagian mana
   yang belum tersedia informasinya.

### CONTEXT
{context}

### FORMAT
- Jawab langsung ke inti dalam 2-4 kalimat. Gunakan poin bernomor jika jawabannya berupa langkah-langkah.
- Akhiri dengan baris "Sumber: [FAQ #id]" berisi id FAQ yang kamu gunakan
  (tulis "Sumber: -" jika tidak ada FAQ yang relevan).

### PERTANYAAN PELANGGAN
{query}

### JAWABAN
'''


def build_context(search_results):
    return '\n\n'.join(
        f"[FAQ #{r['faq_id']}] (kategori: {r['category']}, similarity: {r['similarity']:.3f})\n"
        f"Pertanyaan: {r['question']}\n"
        f"Jawaban: {r['answer']}"
        for r in search_results
    )


def build_prompt(query, search_results):
    return PROMPT_TEMPLATE.format(context=build_context(search_results), query=query)


# Contoh prompt yang dikirim ke LLM
print(build_prompt('kenapa paket saya belum sampai?', test_results))

### PERSONA
Kamu adalah "Asisten CS", asisten virtual customer service sebuah platform e-commerce di Indonesia.
Gaya bahasamu ramah, sopan, jelas, dan menggunakan Bahasa Indonesia yang mudah dipahami pelanggan.

### TASK
Jawab pertanyaan pelanggan HANYA berdasarkan informasi pada bagian CONTEXT di bawah.
Aturan:
1. Gunakan hanya fakta yang tertulis di CONTEXT. Jangan menambahkan informasi dari pengetahuan umum,
   jangan mengarang angka, durasi, biaya, nama fitur, atau kebijakan yang tidak ada di CONTEXT.
2. CONTEXT berisi beberapa FAQ hasil pencarian otomatis dan tidak semuanya pasti relevan.
   Pilih hanya FAQ yang benar-benar menjawab pertanyaan; abaikan yang tidak relevan.
3. Jika CONTEXT tidak memuat informasi untuk menjawab pertanyaan (atau hanya membahas topik yang mirip
   tetapi tidak menjawab pertanyaannya), JANGAN menebak. Jawab dengan:
   "Mohon maaf, saya belum memiliki informasi mengenai hal tersebut di FAQ kami."
   lalu sarankan pelanggan menghubungi customer service.
4

## 7. Mengintegrasikan LLM

`generate_answer()` menggabungkan seluruh alur menjadi satu: **query → retrieval → prompt → jawaban**.

`temperature` dibuat rendah (0.2) agar jawaban konsisten dan tidak "kreatif" di luar context. Jika model pertama di `LLM_MODELS` terkena limit kuota atau tidak tersedia, fungsi otomatis mencoba model berikutnya.

In [11]:
def call_llm(prompt, temperature=0.2):
    """Mengirim prompt ke Gemini. Mencoba model di LLM_MODELS secara berurutan jika gagal."""
    last_error = None
    for model_name in LLM_MODELS:
        try:
            response = with_retry(lambda: client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=temperature),
            ), max_retries=2)
            return response.text.strip(), model_name
        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            print(f'  [fallback] {model_name} gagal ({getattr(e, "code", "")}), mencoba model berikutnya...')
            last_error = e
    raise last_error


def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)
    answer, model_used = call_llm(prompt)
    return answer, search_results


answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan (hasil retrieval):')
for s in sources:
    print(f"- [FAQ #{s['faq_id']}] {s['question']} (similarity={s['similarity']:.3f})")

Jawaban LLM:
Halo! Ya, Anda bisa membayar dengan cicilan. Fitur cicilan tersebut tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

Sumber: [FAQ #20]

Sumber FAQ yang digunakan (hasil retrieval):
- [FAQ #20] Apakah saya bisa membayar dengan cicilan? (similarity=0.807)
- [FAQ #17] Metode pembayaran apa saja yang tersedia? (similarity=0.694)
- [FAQ #22] Apakah bisa membayar menggunakan lebih dari satu metode sekaligus? (similarity=0.688)


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

Setiap query diberi **expected FAQ id** (ground truth yang ditentukan manual dengan membaca knowledge base) sehingga relevansi retrieval bisa diukur secara objektif dengan **Hit@1** (FAQ yang benar ada di peringkat 1) dan **Hit@3** (FAQ yang benar ada di top-3). Untuk query di luar knowledge base, expected = kosong.

In [12]:
test_queries = [
    # --- Tipe 1: query yang jelas ada jawabannya (wording mirip FAQ) ---
    {'type': 'Direct',     'query': 'Bagaimana cara membatalkan pesanan?',                                  'expected': [3]},
    {'type': 'Direct',     'query': 'Metode pembayaran apa saja yang bisa digunakan?',                      'expected': [17]},
    # --- Tipe 2: parafrase / sinonim / bahasa informal ---
    {'type': 'Paraphrase', 'query': 'Barang yang saya beli datang dalam keadaan pecah, saya harus gimana?', 'expected': [16]},
    {'type': 'Paraphrase', 'query': 'Kalau saya retur barang, kapan uang saya balik?',                     'expected': [25]},
    {'type': 'Paraphrase', 'query': 'Saya gak bisa masuk ke akun saya, kenapa ya?',                         'expected': [55]},
    {'type': 'Paraphrase', 'query': 'Bisa bayar pakai GoPay atau OVO nggak?',                               'expected': [17]},
    # --- Tipe 3: query yang TIDAK ada jawabannya di knowledge base ---
    {'type': 'Out-of-KB',  'query': 'Apakah bisa kirim barang ke luar negeri?',                            'expected': []},
    {'type': 'Out-of-KB',  'query': 'Berapa harga iPhone 16 Pro di toko kalian?',                          'expected': []},
]

test_log = []
for i, t in enumerate(test_queries, start=1):
    answer, sources = generate_answer(t['query'], top_k=3)
    retrieved_ids = [s['faq_id'] for s in sources]
    test_log.append({
        'no': i,
        'type': t['type'],
        'query': t['query'],
        'expected': t['expected'],
        'retrieved_top3': retrieved_ids,
        'top1_similarity': sources[0]['similarity'],
        'hit@1': (retrieved_ids[0] in t['expected']) if t['expected'] else None,
        'hit@3': any(fid in t['expected'] for fid in retrieved_ids) if t['expected'] else None,
        'sources': sources,
        'answer': answer,
    })

    print(f"[{i}] ({t['type']}) Query: {t['query']}")
    print(f"Expected FAQ: {t['expected'] or '- (tidak ada di KB)'}")
    print('Hasil retrieval:')
    for s in sources:
        print(f"   #{s['rank']} [FAQ #{s['faq_id']}] sim={s['similarity']:.3f} | {s['question']}")
    print('Jawaban LLM:')
    print('   ' + answer.replace('\n', '\n   '))
    print('-' * 100)

[1] (Direct) Query: Bagaimana cara membatalkan pesanan?
Expected FAQ: [3]
Hasil retrieval:
   #1 [FAQ #3] sim=0.780 | Bagaimana cara membatalkan pesanan yang sudah dibuat?
   #2 [FAQ #24] sim=0.714 | Bagaimana prosedur pengembalian barang yang tidak sesuai?
   #3 [FAQ #8] sim=0.703 | Kenapa pesanan saya otomatis dibatalkan sistem?
Jawaban LLM:
   Halo Kak! Pesanan Kakak dapat dibatalkan dengan mengikuti langkah-langkah berikut:
   
   1. Buka halaman 'Pesanan Saya'.
   2. Lakukan pembatalan selama status pesanan masih 'Diproses'. 
   
   Perlu diketahui bahwa setelah status berubah menjadi 'Dikirim', pembatalan sudah tidak dapat dilakukan ya Kak.
   
   Sumber: [FAQ #3]
----------------------------------------------------------------------------------------------------


[2] (Direct) Query: Metode pembayaran apa saja yang bisa digunakan?
Expected FAQ: [17]
Hasil retrieval:
   #1 [FAQ #17] sim=0.791 | Metode pembayaran apa saja yang tersedia?
   #2 [FAQ #22] sim=0.731 | Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
   #3 [FAQ #20] sim=0.704 | Apakah saya bisa membayar dengan cicilan?
Jawaban LLM:
   Halo! Kami mendukung beberapa metode pembayaran untuk transaksi Anda, yaitu transfer bank, kartu kredit/debit, e-wallet, dan pembayaran melalui minimarket rekanan. 
   
   Sumber: [FAQ #17]
----------------------------------------------------------------------------------------------------


[3] (Paraphrase) Query: Barang yang saya beli datang dalam keadaan pecah, saya harus gimana?
Expected FAQ: [16]
Hasil retrieval:
   #1 [FAQ #16] sim=0.768 | Apa yang harus dilakukan jika paket diterima dalam kondisi rusak?
   #2 [FAQ #54] sim=0.728 | Saya menerima barang yang salah, apa yang harus saya lakukan?
   #3 [FAQ #24] sim=0.715 | Bagaimana prosedur pengembalian barang yang tidak sesuai?
Jawaban LLM:
   Halo! Mohon maaf atas kendala paket yang diterima dalam kondisi pecah atau rusak. Berikut adalah langkah-langkah yang harus Anda lakukan:
   
   1. Segera ambil foto atau video kondisi paket saat diterima.
   2. Laporkan melalui menu komplain dalam waktu 2x24 jam sejak barang diterima.
   
   Sumber: [FAQ #16]
----------------------------------------------------------------------------------------------------


[4] (Paraphrase) Query: Kalau saya retur barang, kapan uang saya balik?
Expected FAQ: [25]
Hasil retrieval:
   #1 [FAQ #25] sim=0.789 | Berapa lama proses refund setelah pengembalian disetujui?
   #2 [FAQ #27] sim=0.732 | Ke mana dana refund akan dikembalikan?
   #3 [FAQ #24] sim=0.729 | Bagaimana prosedur pengembalian barang yang tidak sesuai?
Jawaban LLM:
   Halo! Dana refund biasanya akan diproses dalam waktu 3-7 hari kerja setelah barang dikembalikan diterima dan diverifikasi oleh penjual. Dana tersebut akan dikembalikan ke metode pembayaran awal atau ke saldo akun, tergantung pada opsi yang Anda pilih saat pengajuan.
   
   Sumber: [FAQ #25], [FAQ #27]
----------------------------------------------------------------------------------------------------


[5] (Paraphrase) Query: Saya gak bisa masuk ke akun saya, kenapa ya?
Expected FAQ: [55]
Hasil retrieval:
   #1 [FAQ #55] sim=0.792 | Kenapa akun saya tiba-tiba tidak bisa login?
   #2 [FAQ #32] sim=0.743 | Saya lupa password akun, bagaimana cara mengaturnya ulang?
   #3 [FAQ #34] sim=0.686 | Apakah bisa memiliki lebih dari satu akun dengan email yang sama?
Jawaban LLM:
   Halo! Akun Anda dapat terkunci sementara karena adanya kesalahan login berulang atau terindikasi adanya aktivitas mencurigakan. Untuk mengatasinya, silakan lakukan reset password atau hubungi customer service kami.
   
   Sumber: [FAQ #55]
----------------------------------------------------------------------------------------------------


[6] (Paraphrase) Query: Bisa bayar pakai GoPay atau OVO nggak?
Expected FAQ: [17]
Hasil retrieval:
   #1 [FAQ #17] sim=0.721 | Metode pembayaran apa saja yang tersedia?
   #2 [FAQ #22] sim=0.704 | Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
   #3 [FAQ #37] sim=0.666 | Bagaimana cara menggunakan kode voucher saat checkout?
Jawaban LLM:
   Halo! Platform kami mendukung pembayaran menggunakan e-wallet. Anda dapat memilih opsi tersebut saat melakukan transaksi pembayaran.
   
   Sumber: [FAQ #17]
----------------------------------------------------------------------------------------------------


[7] (Out-of-KB) Query: Apakah bisa kirim barang ke luar negeri?
Expected FAQ: - (tidak ada di KB)
Hasil retrieval:
   #1 [FAQ #15] sim=0.710 | Apakah bisa memilih jasa pengiriman sendiri saat checkout?
   #2 [FAQ #9] sim=0.709 | Berapa lama estimasi waktu pengiriman barang?
   #3 [FAQ #5] sim=0.702 | Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?
Jawaban LLM:
   Mohon maaf, saya belum memiliki informasi mengenai hal tersebut di FAQ kami. Jika Anda membutuhkan bantuan lebih lanjut, silakan hubungi customer service kami.
   
   Sumber: -
----------------------------------------------------------------------------------------------------


[8] (Out-of-KB) Query: Berapa harga iPhone 16 Pro di toko kalian?
Expected FAQ: - (tidak ada di KB)
Hasil retrieval:
   #1 [FAQ #58] sim=0.643 | Apakah bisa memesan dalam jumlah besar untuk kebutuhan reseller?
   #2 [FAQ #1] sim=0.633 | Bagaimana cara melakukan pemesanan di platform ini?
   #3 [FAQ #57] sim=0.621 | Bagaimana cara membandingkan beberapa produk sekaligus?
Jawaban LLM:
   Mohon maaf, saya belum memiliki informasi mengenai hal tersebut di FAQ kami. Silakan menghubungi customer service kami untuk mendapatkan bantuan lebih lanjut.
   
   Sumber: -
----------------------------------------------------------------------------------------------------


### Ringkasan metrik retrieval

In [13]:
summary = pd.DataFrame(test_log)[['no', 'type', 'query', 'expected', 'retrieved_top3', 'top1_similarity', 'hit@1', 'hit@3']]
summary

,no,type,query,expected,retrieved_top3,top1_similarity,hit@1,hit@3
0,1,Direct,Bagaimana cara membatalkan pesanan?,[3],"[3, 24, 8]",0.7796,True,True
1,2,Direct,Metode pembayaran apa saja yang bisa digunakan?,[17],"[17, 22, 20]",0.7911,True,True
2,3,Paraphrase,Barang yang saya beli datang dalam keadaan pec...,[16],"[16, 54, 24]",0.7678,True,True
3,4,Paraphrase,"Kalau saya retur barang, kapan uang saya balik?",[25],"[25, 27, 24]",0.7895,True,True
4,5,Paraphrase,"Saya gak bisa masuk ke akun saya, kenapa ya?",[55],"[55, 32, 34]",0.7923,True,True
5,6,Paraphrase,Bisa bayar pakai GoPay atau OVO nggak?,[17],"[17, 22, 37]",0.7210,True,True
6,7,Out-of-KB,Apakah bisa kirim barang ke luar negeri?,[],"[15, 9, 5]",0.7104,None,None
7,8,Out-of-KB,Berapa harga iPhone 16 Pro di toko kalian?,[],"[58, 1, 57]",0.6432,None,None


In [14]:
in_kb = summary[summary['type'] != 'Out-of-KB']
print(f"Query yang ada di KB : {len(in_kb)}")
print(f"Hit@1                : {in_kb['hit@1'].sum()}/{len(in_kb)} ({in_kb['hit@1'].mean():.0%})")
print(f"Hit@3                : {in_kb['hit@3'].sum()}/{len(in_kb)} ({in_kb['hit@3'].mean():.0%})")
print()
print('Rata-rata top-1 similarity per tipe query:')
print(summary.groupby('type', sort=False)['top1_similarity'].agg(['mean', 'min', 'max']).round(3))

Query yang ada di KB : 6
Hit@1                : 6/6 (100%)
Hit@3                : 6/6 (100%)

Rata-rata top-1 similarity per tipe query:
             mean    min    max
type                           
Direct      0.785  0.780  0.791
Paraphrase  0.768  0.721  0.792
Out-of-KB   0.677  0.643  0.710


### Menyimpan hasil pengujian

Hasil mentah pengujian (query, hasil retrieval, similarity, jawaban LLM) disimpan ke `documentation/retrieval_test_results.md` sebagai lampiran analisis di README.

In [15]:
DOC_PATH = PROJECT_ROOT / 'documentation' / 'retrieval_test_results.md'
DOC_PATH.parent.mkdir(exist_ok=True)

lines = [
    '# Retrieval Test Results',
    '',
    f'- Embedding model: `{EMBEDDING_MODEL}` (dimensi {dimension})',
    f'- Vector DB: FAISS `IndexFlatIP` (cosine similarity), {index.ntotal} vektor',
    '- top_k = 3',
    '',
    '> Analisis relevansi hasil pengujian ini ada di [README.md](../README.md#3-hasil-pengujian--analisis-relevansi) dan di bagian akhir notebook.',
    '',
    '## Ringkasan',
    '',
    '| No | Tipe | Query | Expected FAQ | Retrieved top-3 | Top-1 sim | Hit@1 | Hit@3 |',
    '|---|---|---|---|---|---|---|---|',
]
for r in test_log:
    fmt = lambda v: '-' if v is None else ('✅' if v else '❌')
    lines.append(
        f"| {r['no']} | {r['type']} | {r['query']} | {r['expected'] or '-'} | {r['retrieved_top3']} | "
        f"{r['top1_similarity']:.3f} | {fmt(r['hit@1'])} | {fmt(r['hit@3'])} |"
    )
lines += ['', '## Detail per query', '']
for r in test_log:
    lines += [f"### {r['no']}. [{r['type']}] {r['query']}", '',
              f"Expected FAQ: {r['expected'] or '- (tidak ada di knowledge base)'}", '',
              '| Rank | FAQ id | Kategori | Pertanyaan FAQ | Similarity |', '|---|---|---|---|---|']
    for s in r['sources']:
        lines.append(f"| {s['rank']} | {s['faq_id']} | {s['category']} | {s['question']} | {s['similarity']:.3f} |")
    lines += ['', '**Jawaban LLM:**', '', '> ' + r['answer'].replace('\n', '\n> '), '']

DOC_PATH.write_text('\n'.join(lines), encoding='utf-8')
print('Hasil pengujian disimpan ke:', DOC_PATH.relative_to(PROJECT_ROOT))

Hasil pengujian disimpan ke: documentation\retrieval_test_results.md


### Catatan Analisis Relevansi

_Tuliskan pengamatanmu di sini: query mana yang berhasil di-retrieve dengan baik, query mana yang kurang relevan hasilnya, dan bagaimana LLM merespons ketika informasi tidak tersedia di knowledge base._

**Ringkasan:** retrieval mengembalikan FAQ yang benar di peringkat 1 untuk **6/6 query** yang memang ada di knowledge base (Hit@1 = 100%), termasuk 4 query parafrase/informal. Kedua query di luar knowledge base dijawab dengan penolakan yang benar oleh LLM.

> Catatan: angka similarity dan teks jawaban di bawah berasal dari run notebook ini. Jawaban LLM bisa sedikit berbeda wording-nya di setiap run, walaupun `temperature` sudah rendah.

**1. Query direct (#1, #2), retrieval akurat.**
FAQ #3 dan #17 berada di rank 1 dengan similarity ±0.78–0.79. Ini hasil yang wajar karena wording query hampir sama dengan pertanyaan FAQ. Kandidat rank 2–3 masih satu topik tapi tidak menjawab pertanyaan; misalnya pada query #1, FAQ #24 (pengembalian barang) dan #8 (pembatalan otomatis oleh sistem). LLM mengabaikan kandidat tersebut dan hanya mengutip FAQ #3, sesuai aturan nomor 2 di prompt.

**2. Query parafrase (#3–#6), semantic search menangkap makna, bukan kata kunci.**
- #3 *"datang dalam keadaan **pecah**"* → FAQ #16 *"diterima dalam kondisi **rusak**"*. Tidak ada kata kunci yang sama ("pecah" vs "rusak", "datang" vs "diterima"), tetapi FAQ tetap ter-retrieve di rank 1.
- #4 *"**retur** barang, kapan **uang saya balik**"* → FAQ #25 *"proses **refund** setelah **pengembalian**"*. Kosakata informal/serapan dipetakan ke istilah formal. LLM menggabungkan FAQ #25 (durasi) dan #27 (tujuan dana) menjadi satu jawaban yang lebih lengkap, sehingga top-k > 1 terbukti berguna.
- #5 *"gak bisa **masuk ke akun**"* → FAQ #55 *"tidak bisa **login**"*. Bahasa gaul dan sinonim lintas bahasa (masuk ↔ login) tertangkap.
- #6 *"bayar pakai **GoPay atau OVO**"* → FAQ #17 (metode pembayaran, menyebut "e-wallet"). Retrieval benar, tetapi similarity-nya paling rendah di antara query in-KB (0.721) karena model harus tahu bahwa GoPay/OVO adalah e-wallet. **Jawaban LLM kurang presisi:** FAQ hanya menyebut "e-wallet" secara umum, tidak menyebut merek tertentu. Idealnya LLM menjelaskan bahwa e-wallet didukung tetapi FAQ tidak merinci e-wallet mana saja (aturan nomor 4: jawaban parsial). Jawaban saat ini tidak salah, tetapi bisa diartikan pelanggan sebagai konfirmasi bahwa GoPay/OVO pasti bisa dipakai.

**3. Query di luar KB (#7, #8), retrieval tetap mengembalikan hasil, LLM yang menjadi penjaga.**
- #7 *"kirim ke luar negeri"* menghasilkan FAQ pengiriman (#15, #9, #5) dengan similarity 0.70–0.71. Topiknya mirip tetapi tidak menjawab. LLM tidak tergoda menyimpulkan dari FAQ #9 ("luar pulau") dan menjawab dengan kalimat penolakan baku, `Sumber: -`.
- #8 *"harga iPhone"* menghasilkan similarity paling rendah (0.643) dan LLM menolak dengan benar.
- **Temuan penting:** FAISS selalu mengembalikan top-k, berapa pun skornya. Rentang similarity juga **sempit dan saling tumpang tindih**: query out-of-KB #7 (0.710) hampir sama dengan query in-KB #6 (0.721). Artinya **threshold similarity saja tidak cukup** untuk mendeteksi pertanyaan di luar knowledge base. Kalau threshold dipasang di 0.72, query #6 yang valid ikut terbuang. Pada prototype ini, instruksi di prompt (aturan nomor 3) yang menjadi penjaga utama terhadap halusinasi, dan cara ini berhasil pada kedua kasus.

**4. Kualitas jawaban LLM secara umum.**
Semua jawaban memakai fakta dari FAQ tanpa menambah angka atau kebijakan baru, contohnya "2x24 jam", "3–7 hari kerja", dan status "Diproses"/"Dikirim". Jawaban yang berupa langkah-langkah otomatis diformat sebagai poin bernomor, dan setiap jawaban mencantumkan sumber `[FAQ #id]` sehingga mudah diaudit.